# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## Research Question

Which observable content and search-performance signals can help prioritize pages for human content-refresh review?

### Decision supported

The analysis is designed to help a content team decide which pages should be reviewed first for a possible refresh. It provides a ranked review queue using an ML model and observable signals such as click-through rate, impressions, engagement, search position, content age, and word count.

The model is used as a decision-support tool. A high score does not mean that a page definitely needs a refresh, and the analysis does not claim that any individual signal causes content decline.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

This project uses the anonymized FlyRank search/content dataset provided for the Machine Learning internship.

The dataset contains page-level content and search-performance signals used to study content refresh opportunities. The original dataset contains 30,000 rows and 44 columns. After creating the `is_declining_label` target column, the working dataframe contains 45 columns.

The analysis uses observable features including search volume, competition, CPC, word count, character count, 90-day impressions, 90-day clicks, pageviews, sessions, CTR, average search position, engagement rate, scroll rate, AI traffic percentage, days since last update, content type, search intent, competition level, impression tier, and position tier.

The target label is `is_declining_label`, created from the dataset's `trend_direction` field. A page is labelled 1 when its trend direction is `down`, and 0 otherwise.

The modelling features exclude `trend_direction` and `trend_pct` because these fields are directly related to the target definition and could introduce leakage.

Client-identifying information, private queries, URLs, credentials, and raw sensitive exports are not included in the public analysis.

The model is intended to use observable signals available in the dataset to prioritize pages for human review rather than to establish causal relationships.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Methodology

### Feature selection

The model uses numerical and categorical content/search-performance features.

Numerical features include:

- search_volume
- competition
- cpc
- word_count
- char_count
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- days_since_last_update

Categorical features include:

- content_type
- main_intent
- competition_level
- impression_tier
- position_tier

The fields `trend_direction` and `trend_pct` are excluded from the model because they are connected to the target definition and could leak information about the outcome.

### Model

A Random Forest classifier was selected because it can work with nonlinear relationships and combinations of multiple content and search-performance signals. Categorical variables are encoded before modelling and numerical variables are handled through the preprocessing pipeline.

### Label

The target is `is_declining_label`.

A value of 1 represents a page whose observed trend direction is `down`, while 0 represents the other observed trend directions.

### Validation

The grouped-by-client split produced 23,837 training rows and 6,163 test rows. The training set contained 25 clients and the test set contained 7 previously unseen clients.

This means that pages from the same client were not split between training and testing.

### Decision output

The trained model produces a probability-like score representing the estimated likelihood that a page belongs to the declining-content class.

Pages are ranked by this score to create a review queue.

Reason codes are then added using observable feature comparisons with the training data. These codes explain signals associated with a page, but they are not interpreted as causal explanations.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Results

The same Random Forest model and feature set were evaluated using two validation designs.

| Metric | Random row split (Before) | Grouped-by-client (After) |
|---|---:|---:|
| Precision@20 | 1.000 | 0.750 |
| Precision@50 | 0.980 | 0.720 |
| Precision@100 | 0.970 | 0.680 |
| ROC AUC | 0.743 | 0.637 |

The random row split produced Precision@20 of 1.000, Precision@50 of 0.980, Precision@100 of 0.970, and ROC AUC of 0.743.

The grouped-by-client split produced Precision@20 of 0.750, Precision@50 of 0.720, Precision@100 of 0.680, and ROC AUC of 0.637.

The lower grouped-by-client results show that validation design has a substantial effect on measured performance. Because the grouped split evaluates the model on previously unseen clients, I treat it as the more relevant estimate of generalization for this project.

The grouped-by-client results are therefore the more appropriate results to use for the final decision-support workflow.

These results should be interpreted as decision-support evidence rather than proof that the model identifies the causes of content decline or predicts Google's ranking algorithm.

## 5. Limitations

*What this work cannot claim.*

## Limitations

This analysis has several limitations.

First, the model identifies patterns associated with the observed declining-content label. It does not establish that any feature causes a page to decline.

Second, the validation evaluates generalization to unseen clients within the available dataset. It does not prove that the model will perform the same way on future datasets or other organizations.

Third, the ranked queue is a prioritization tool rather than an automatic content-refresh system. A high model score should trigger human review rather than an automatic rewrite, deletion, or publishing decision.

Fourth, the reason codes are based on observable feature signals and simple comparisons with the training data. They should not be interpreted as explanations of why a page declined.

Finally, search performance can change over time, so the model and its thresholds should be monitored and re-evaluated when the underlying data distribution changes.The dataset represents a specific set of anonymized pages and clients, so the observed performance should not be assumed to generalize to every website, industry, or future dataset without additional validation.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Ranked Recommendations

The final action playbook converts model scores into a ranked content-refresh review queue.

The highest-scoring pages are presented first because they have the highest model scores for the declining-content class.

Each page is also assigned reason codes based on observable signals.

Examples include:

- `LOW_CTR` — relatively low click-through rate
- `LOW_IMPRESSIONS` — relatively low impressions
- `LOW_ENGAGEMENT` — relatively low engagement
- `OLD_CONTENT` — older content based on days since last update
- `LOW_POSITION` — relatively weak average search position
- `HIGH_WORD_COUNT` — relatively high word count

These reason codes are intended to help a human reviewer understand which observable signals contributed to the review priority.

The queue does not automatically recommend rewriting or removing content. A human should review the page, search intent, content quality, business context, and other relevant evidence before taking action.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Reproducibility

The analysis is organized as a sequence of notebooks in the internship repository.

The main workflow is documented in:

- `work/notebooks/w04_signal_audit.ipynb`
- `work/notebooks/w05_model.ipynb`
- `work/notebooks/w06_validation_audit.ipynb`
- `work/notebooks/w07_action_playbook.ipynb`
- `work/notebooks/capstone.ipynb`

The workflow covers signal checks, model training, grouped-by-client validation, evaluation, and ranked action recommendations.

The Week 7 action playbook produces a ranked review queue containing the model score and observable reason codes. The queue is exported to:

`work/outputs/content_refresh_action_queue.csv`

The main model results reported in this capstone are:

| Metric | Grouped-by-client result |
|---|---:|
| Precision@20 | 0.750 |
| Precision@50 | 0.720 |
| Precision@100 | 0.680 |
| ROC AUC | 0.637 |

The grouped validation used 23,837 training rows from 25 clients and 6,163 test rows from 7 previously unseen clients.

The notebook workflow and exported queue provide the main artifacts needed to reproduce the analysis and inspect the decision-support output.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.